In [1]:
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity

In [2]:

tfidf_matrix = sp.load_npz('../data/processed/tfidf_matrix.npz')
df_meta = pd.read_csv('../data/processed/vacancies_meta.csv')

# словарь для быстрого поиска
id_to_index = pd.Series(df_meta.index, index=df_meta['vacancy_id']).to_dict()

In [3]:
def get_recommendations(target_vacancy_id: int, top_k: int = 5) -> pd.DataFrame:

    if target_vacancy_id not in id_to_index:
        return "Ошибка: Вакансия с таким ID не найдена."
    
    idx = id_to_index[target_vacancy_id]
    
    target_vector = tfidf_matrix[idx]
    
    similarities = cosine_similarity(target_vector, tfidf_matrix).flatten()
    
    # 5. Сортируем индексы, Убираем саму себя из результатов
    related_docs_indices = similarities.argsort()[-2 : -(top_k + 2) : -1]
    
    result_df = df_meta.iloc[related_docs_indices].copy()
    
    # Добавляем колонку со score (в процентах)
    result_df['similarity_score'] = similarities[related_docs_indices].round(3)
    
    return result_df

In [4]:
random_row = df_meta.sample(n=1)

target_id = random_row['vacancy_id'].iloc[0]
print("Ищем похожие на:")
display(df_meta[df_meta['vacancy_id'] == target_id])

print("\nРекомендации:")
recommendations = get_recommendations(target_id, top_k=5)
display(recommendations)

rec_ids = recommendations['vacancy_id'].tolist()
all_ids = [int(target_id)] + rec_ids

print("\n--- Строка для копирования ---")
print(f"ids_to_check = {all_ids}")

Ищем похожие на:


,vacancy_id,title,author_name,salary_min,salary_max,city,experience_min,remote_type
25100,49463270,Business development manager (junior),Меркурио,80000.0,110000.0,Москва,1,OFFICE



Рекомендации:


,vacancy_id,title,author_name,salary_min,salary_max,city,experience_min,remote_type,similarity_score
620,49767087,Business Development Manager (junior),AdChampagne,50000.0,100000.0,Санкт-Петербург,0,OFFICE,0.422
37081,47096652,Project manager (FinTech),Меркурио,NaN,NaN,Москва,1,REMOTE,0.398
621,49764824,Business Development Manager (senior),AdChampagne,150000.0,NaN,Санкт-Петербург,3,OFFICE,0.356
5781,47979160,Business analyst,Меркурио,150000.0,230000.0,Москва,1,REMOTE,0.341
11606,50015216,Product manager,Меркурио,NaN,NaN,Москва,3,REMOTE,0.341



--- Строка для копирования ---
ids_to_check = [49463270, 49767087, 47096652, 49764824, 47979160, 50015216]
